# Bab 13: Jaringan Saraf Tiruan dan Pengantar Deep Learning

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab13/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab13")
print("Folder kerja:", os.getcwd())

## Listing 13.1: Perceptron untuk AND dan XOR

`kode/bab13/perceptron.py`

In [ ]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
data = {"AND": np.array([0, 0, 0, 1]), "XOR": np.array([0, 1, 1, 0])}


def latih(X, y, eta=0.5, epoch=20, cetak=False):
    w, b = np.zeros(2), 0.0
    for e in range(1, epoch + 1):
        salah = 0
        for xi, yi in zip(X, y):
            yhat = int(w @ xi + b > 0)
            if yhat != yi:
                w += eta * (yi - yhat) * xi
                b += eta * (yi - yhat)
                salah += 1
            if cetak and e <= 2:
                print(f"  x={xi} y={yi} yhat={yhat} -> w={w} b={b:+.1f}")
        if salah == 0:
            return w, b, e
    return w, b, None


for nama, y in data.items():
    print(nama)
    w, b, e = latih(X, y, cetak=(nama == "AND"))
    if e:
        print(f"  konvergen pada epoch {e}: w={w}, b={b:+.1f}")
    else:
        print("  tidak konvergen setelah 20 epoch")

## Listing 13.2: Satu iterasi backpropagation, lalu pelatihan penuh XOR

`kode/bab13/xor_backprop.py`

In [ ]:
import numpy as np

sig = lambda z: 1 / (1 + np.exp(-z))
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y = np.array([0, 1, 1, 0], float)
W1 = np.array([[0.5, -0.5], [0.5, -0.5]])
b1 = np.array([-0.2, 0.7])
w2 = np.array([1.0, 1.0])
b2 = -1.0
eta = 0.5

x, t = X[2], y[2]
z1 = x @ W1 + b1; h = sig(z1)
z2 = h @ w2 + b2; o = sig(z2)
print("maju : z1 =", z1.round(4), " h =", h.round(4))
print(f"       z2 = {z2:.4f}  o = {o:.4f}  galat = {0.5*(t-o)**2:.4f}")
d2 = (o - t) * o * (1 - o)
d1 = d2 * w2 * h * (1 - h)
print(f"mundur: delta2 = {d2:.4f}  delta1 =", d1.round(4))
w2 -= eta * d2 * h; b2 -= eta * d2
W1 -= eta * np.outer(x, d1); b1 -= eta * d1
print("baru : w2 =", w2.round(4), f" b2 = {b2:.4f}")
print("       W1 =", W1.round(4).tolist(), " b1 =", b1.round(4))
o_baru = sig(sig(x @ W1 + b1) @ w2 + b2)
print(f"       o baru = {o_baru:.4f} (target {t:.0f})")

rng = np.random.default_rng(3)
W1, b1 = rng.normal(0, 1, (2, 2)), np.zeros(2)
w2, b2 = rng.normal(0, 1, 2), 0.0
for ep in range(1, 5001):
    H = sig(X @ W1 + b1); O = sig(H @ w2 + b2)
    D2 = (O - y) * O * (1 - O)
    D1 = np.outer(D2, w2) * H * (1 - H)
    w2 -= eta * H.T @ D2; b2 -= eta * D2.sum()
    W1 -= eta * X.T @ D1; b1 -= eta * D1.sum(0)
    if ep in (1, 1000, 5000):
        print(f"epoch {ep:4d}: galat {0.5*((y-O)**2).sum():.4f}"
              f"  o = {O.round(2)}")

## Listing 13.3: MLP scikit-learn untuk belanja bulan depan

`kode/bab13/mlp_skala.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab13')
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sys
sys.path.insert(0, "kode/bab12")
from siapkan import DATA, KATEGORIK, NUMERIK

df = pd.read_csv(DATA + "pelanggan.csv")
X, y = df[NUMERIK + KATEGORIK], df["belanja_bulan_depan"] / 1000
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, random_state=7)


def pipa(skala, lapisan, alpha=1e-4, solver="adam", skala_y=True):
    num = StandardScaler() if skala else "passthrough"
    m = make_pipeline(ColumnTransformer(
        [("num", num, NUMERIK),
         ("kat", OneHotEncoder(handle_unknown="ignore"), KATEGORIK)]),
        MLPRegressor(hidden_layer_sizes=lapisan, alpha=alpha, solver=solver,
                     learning_rate_init=1e-3,
                     max_iter=500, early_stopping=True,
                     n_iter_no_change=20, random_state=0))
    if not skala_y:
        return m
    return TransformedTargetRegressor(m, transformer=StandardScaler())


for solver, skala, sy in [("sgd", False, False), ("sgd", True, False),
                          ("sgd", True, True), ("adam", True, True)]:
    m = pipa(skala, (32,), solver=solver, skala_y=sy).fit(Xl, yl)
    print(f"{solver:4s} skala X={skala!s:5s} skala y={sy!s:5s}: MAE uji "
          f"{mean_absolute_error(yu, m.predict(Xu)):7.1f}")
print()
for lap in [(4,), (32,), (128,), (64, 32), (128, 64, 32)]:
    m = pipa(True, lap).fit(Xl, yl)
    mlp = m.regressor_[-1]
    n = sum(w.size for w in mlp.coefs_) + sum(
        b.size for b in mlp.intercepts_)
    print(f"lapisan {str(lap):13s} parameter {n:6d}  MAE uji "
          f"{mean_absolute_error(yu, m.predict(Xu)):5.1f}")

m = pipa(True, (64, 32)).fit(Xl, yl)
with open("gambar/bab13/kurva_latih.dat", "w") as f:
    f.write("epoch loss r2val\n")
    mlp = m.regressor_[-1]
    for i, (l, v) in enumerate(zip(mlp.loss_curve_,
                                   mlp.validation_scores_), 1):
        f.write(f"{i} {l:.4f} {v:.4f}\n")

## Listing 13.4: Jaringan PyTorch dengan embedding gerai dan dua kepala

`kode/bab13/pytorch_dua_kepala.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab13')
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from torch import nn
from siapkan import data_tensor

torch.manual_seed(0)
torch.set_num_threads(2)
df, X, G, y, latih, uji = data_tensor()


class Belanja(nn.Module):
    def __init__(self, p, n_gerai=40, d_emb=3):
        super().__init__()
        self.emb = nn.Embedding(n_gerai, d_emb)
        self.badan = nn.Sequential(
            nn.Linear(p + d_emb, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU())
        self.kepala_beli = nn.Linear(32, 1)
        self.kepala_nilai = nn.Linear(32, 1)

    def forward(self, x, g):
        z = self.badan(torch.cat([x, self.emb(g)], dim=1))
        return (self.kepala_beli(z).squeeze(1),
                nn.functional.softplus(self.kepala_nilai(z)).squeeze(1))


def galat(logit, nilai, y):
    beli = (y > 0).float()
    l1 = nn.functional.binary_cross_entropy_with_logits(logit, beli)
    l2 = (nn.functional.huber_loss(nilai, y, delta=50.0,
                                   reduction="none") * beli).sum() / beli.sum()
    return l1 + l2 / 100


model = Belanja(X.shape[1])
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
il, iv = latih[:4800], latih[4800:]
terbaik, sabar, simpan = np.inf, 0, None
for ep in range(1, 301):
    model.train()
    for i in torch.randperm(len(il)).split(128):
        b = torch.tensor(il)[i]
        opt.zero_grad()
        loss = galat(*model(X[b], G[b]), y[b])
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        lv = galat(*model(X[iv], G[iv]), y[iv]).item()
    if lv < terbaik - 1e-4:
        terbaik, sabar = lv, 0
        simpan = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        sabar += 1
        if sabar == 20:
            break
model.load_state_dict(simpan)
with torch.no_grad():
    logit, nilai = model(X[uji], G[uji])
    p = torch.sigmoid(logit)
    yhat = (p * nilai).numpy()
yu = y[uji].numpy()
print(f"berhenti pada epoch {ep}, galat validasi terbaik {terbaik:.4f}")
print(f"MAE uji {np.abs(yu - yhat).mean():.1f} ribu Rp")
print(f"AUC 'akan belanja': {roc_auc_score(yu > 0, p):.3f}")
E = model.emb.weight.detach().numpy()
np.save("kode/bab13/prediksi_torch.npy", yhat)
np.savetxt("kode/bab13/embedding_gerai.txt", E, fmt="%.4f")
print("embedding gerai G01..G03:\n", E[:3].round(2))

## Listing 13.5: Studi kasus perbandingan model belanja bulan depan

`kode/bab13/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab13')
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from siapkan import DATA, KATEGORIK, NUMERIK

warnings.filterwarnings("ignore")
df = pd.read_csv(DATA + "pelanggan.csv")
X, y = df[NUMERIK + KATEGORIK], df["belanja_bulan_depan"] / 1000
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, random_state=7)
pra = lambda: ColumnTransformer(
    [("num", StandardScaler(), NUMERIK),
     ("kat", OneHotEncoder(handle_unknown="ignore"), KATEGORIK)])
model = {
    "linier": make_pipeline(pra(), LinearRegression()),
    "LightGBM": make_pipeline(pra(), lgb.LGBMRegressor(
        n_estimators=400, learning_rate=0.03, num_leaves=15,
        min_child_samples=40, verbose=-1, random_state=0)),
    "MLP (4,)": TransformedTargetRegressor(make_pipeline(
        pra(), MLPRegressor(hidden_layer_sizes=(4,), max_iter=500,
                            early_stopping=True, n_iter_no_change=20,
                            random_state=0)),
        transformer=StandardScaler())}
pred = {n: m.fit(Xl, yl).predict(Xu) for n, m in model.items()}
pred["PyTorch 2 kepala"] = np.load("kode/bab13/prediksi_torch.npy")

tk = df.loc[Xu.index, "tingkat"]
print("model             MAE   RMSE  nol-salah  total(jt) galat-total")
for n, p in pred.items():
    p = np.clip(p, 0, None)
    nol = ((p < 20) & (yu > 0)).mean()
    tot = p.sum() / 1000
    print(f"{n:16s} {mean_absolute_error(yu, p):5.1f} "
          f"{np.sqrt(mean_squared_error(yu, p)):6.1f} {nol:9.1%} "
          f"{tot:9.1f} {tot / (yu.sum() / 1000) - 1:+10.1%}")
print(f"{'aktual':16s} {'':5s} {'':6s} {'':9s} {yu.sum() / 1000:9.1f}")

print("\ngalat per tingkat (MAE, ribu Rp):")
tab = pd.DataFrame({n: (np.abs(yu - np.clip(p, 0, None))
                        .groupby(tk).mean()) for n, p in pred.items()})
tab["n"] = tk.value_counts()
tab["rata2 aktual"] = yu.groupby(tk).mean()
print(tab.round(1).to_string())

gab = (pred["linier"] + pred["PyTorch 2 kepala"]) / 2
print(f"\nrata-rata linier + PyTorch: MAE "
      f"{mean_absolute_error(yu, np.clip(gab, 0, None)):.1f}")